# FinSight — Behavioral Finance Synthetic Data Generator

Notebook ini membangkitkan data sintetis mutasi rekening untuk melatih model anomaly detection.
Data dikontrol oleh distribusi probabilitas dan psikologi keuangan (Behavioral Finance), bukan random biasa.

## Parameter

- **Nasabah:** 500 user — 40% Mahasiswa, 40% First Jobber, 20% Profesional
- **Durasi:** 90 hari (Januari–Maret 2026)
- **Output:** `df_nasabah`, `df_transaksi`, `training_data_autoencoder.csv`
- **Seed:** `np.random.seed(42)` dan `Faker.seed(42)`

## Rules Engine

1. **Frekuensi (Poisson):** Jumlah transaksi harian per kategori diatur `lambda` per persona.
2. **Dynamic Throttling:** Rem otomatis berdasarkan rasio `sisa_saldo / gaji` — Sultan (>40%), Panik (15–40%), Survival (<15%).
3. **Cause-and-Effect Persona:** 15% nasabah `is_dynamic`; persona turun jika survival streak >= 7 hari.
4. **Peak Season & Windfall:** Multiplier di tgl 1–5, tanggal kembar, dan 2% peluang/hari pemasukan dadakan.
5. **Injeksi Noise:** Fee admin acak 30%, anomali markup 3x–8x (2%), typo NLP pada catatan P2P (15%).

## Skema DataFrame

**df_nasabah:** `id_user`, `nama_nasabah`, `tanggal_lahir`, `nama_ibu_kandung`, `segmen_demografi`, `gaji_bulanan`, `persona_dasar`, `is_dynamic`, `persona_bulan_1/2/3`

**df_transaksi:** `id_transaksi`, `id_user`, `timestamp`, `tipe_mutasi`, `deskripsi_mutasi`, `catatan_mutasi`, `mcc`, `nominal`, `sisa_saldo`, `kategori_besar_sistem`, `kategori_detail_sistem`, `label_anomali`, `gt_kategori_besar`, `gt_kategori_detail`

In [10]:
import pandas as pd
import numpy as np
!pip install faker
from faker import Faker
import random
from datetime import datetime, timedelta

# Profil Nasabah

In [18]:

!pip install faker
from faker import Faker # Added import for Faker
import random # Added import for random
import numpy as np import os

OUTPUT_DIR = '/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data/new'
os.makedirs(OUTPUT_DIR, exist_ok=True)
fake = Faker('id_ID')
Faker.seed(42)
random.seed(42)
np.random.seed(42)

print("Membangkitkan data nasabah...")

jumlah_nasabah = 500
data_nasabah = []

# porsi 40-40-20
for i in range(jumlah_nasabah):
    id_user = f"USR-{str(i+1).zfill(3)}"
    nama = fake.name()
    tanggal_lahir = fake.date_of_birth(minimum_age=18, maximum_age=35).strftime('%Y-%m-%d')
    nama_ibu_kandung = fake.name_female()

    #  ALOKASI DEMOGRAFI & GAJI
    if i < 200: # 40% Mahasiswa (200 orang)
        segmen = 'Mahasiswa'
        pendapatan = random.randint(150, 350) * 10000 # Rp 1.5 Juta - Rp 3.5 Juta
    elif i < 400: # 40% First Jobber (200 orang)
        segmen = 'First Jobber'
        pendapatan = random.randint(450, 600) * 10000 # Rp 4.5 Juta - Rp 6.0 Juta
    else: # 20% Profesional (100 orang)
        segmen = 'Profesional'
        pendapatan = random.randint(800, 2000) * 10000 
        
    base_persona = random.choice(['Spendthrift', 'Unconflicted', 'Tightwad'])

    # 15% nasabah is_dynamic
    is_dynamic = 1 if random.random() < 0.15 else 0

    data_nasabah.append([
        id_user, nama, tanggal_lahir, nama_ibu_kandung, segmen, pendapatan,
        base_persona, is_dynamic,
        base_persona, base_persona, base_persona # Inisialisasi persona untuk bulan 1, 2, 3
    ])


df_nasabah = pd.DataFrame(data_nasabah, columns=[
    'id_user', 'nama_nasabah', 'tanggal_lahir', 'nama_ibu_kandung',
    'segmen_demografi', 'gaji_bulanan', 'persona_dasar', 'is_dynamic',
    'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3' 
])


df_nasabah = df_nasabah.sample(frac=1, random_state=42).reset_index(drop=True)

print("df_nasabah selesai.\n")
display(df_nasabah.head(20))

print("\n RANGKUMAN DEMOGRAFI:")
print(df_nasabah['segmen_demografi'].value_counts())
print("\n RANGKUMAN PERSONA DINAMIS (1 = Bisa Berubah, 0 = Statis):")
print(df_nasabah['is_dynamic'].value_counts())

SyntaxError: invalid syntax (1128929680.py, line 4)

In [ ]:
print("Jumlah Persona per Bulan 1:")
print(df_nasabah['persona_bulan_1'].value_counts())
print("\nJumlah Persona per Bulan 2:")
print(df_nasabah['persona_bulan_2'].value_counts())
print("\nJumlah Persona per Bulan 3:")
print(df_nasabah['persona_bulan_3'].value_counts())

Jumlah Persona per Bulan 1:
persona_bulan_1
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 2:
persona_bulan_2
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64

Jumlah Persona per Bulan 3:
persona_bulan_3
Spendthrift     174
Unconflicted    167
Tightwad        159
Name: count, dtype: int64


In [13]:
df_nasabah.to_csv('df_nasabah.csv', index=False)
print("df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai 'df_nasabah.csv'")

df_nasabah final (termasuk persona bulanan) berhasil disimpan sebagai 'df_nasabah.csv'


# Tahap 2 — Master Dictionary Merchant

## Master Merchant Dictionary

`master_merchant` adalah nested dict tiga level: **Kategori Besar → Kategori Detail → Merchant**.
Setiap merchant punya `mu` (harga rata-rata), `sigma` (deviasi), dan `mcc` (4-digit MCC).
Tersedia merchant kelas atas dan bawah per kategori agar K-Medoids bisa memisahkan persona.

## Arsitektur Klasifikasi MCC

**Jalur 1 — Rule-Based (MCC bank)**

| Kategori Besar | Kategori Detail | MCC |
| :--- | :--- | :--- |
| Needs | Tagihan & Utilitas | `4900` |
| Needs | Transportasi | `4121`, `5541` |
| Needs | Groceries & Kebutuhan Pokok | `5411` |
| Needs | Kesehatan & Perawatan Diri | `5912`, `8099` |
| Wants | F&B dan Nongkrong | `5812`, `5814` |
| Wants | Hiburan & Langganan | `4899`, `7832`, `7999` |
| Wants | Belanja Online & Fashion | `5311`, `5651` |
| Transfer | Transfer P2P | `4829` |
| Transfer | Investasi & Finansial | `6211`, `6012` |

**Jalur 2 — NLP Target (Transfer P2P, MCC 4829)**

Model TF-IDF + Naive Bayes membaca `catatan_mutasi` untuk mengklasifikasikan tujuan transfer (Needs / Wants / Investasi).

## Tahap 2 — Running Syntax

In [14]:
print("Membangun master merchant dictionary...")

master_merchant = {
    'Needs': {
        'Tagihan & Utilitas': {
            'PLN MOBILE': {'mu': 200000, 'sigma': 50000, 'mcc': '4900'},
            'PDAM': {'mu': 100000, 'sigma': 25000, 'mcc': '4900'},
            'INDIHOME': {'mu': 350000, 'sigma': 20000, 'mcc': '4900'},
            'UKT KAMPUS': {'mu': 5000000, 'sigma': 1000000, 'mcc': '8220'},
            'KPR BANK': {'mu': 3000000, 'sigma': 750000, 'mcc': '6012'},
            'PAKET DATA': {'mu': 50000, 'sigma': 25000, 'mcc': '4814'},
            'BIAYA KOSAN BULANAN': {'mu': 1200000, 'sigma': 300000, 'mcc': '6513'},
            'LAUNDRY KILOAN': {'mu': 35000, 'sigma': 10000, 'mcc': '7210'},
            'ICLOUD/GOOGLE ONE': {'mu': 45000, 'sigma': 0, 'mcc': '5734'}
        },
        'Transportasi': {
            'SPBU PERTAMINA': {'mu': 100000, 'sigma': 50000, 'mcc': '5541'},
            'SHELL INDONESIA': {'mu': 200000, 'sigma': 40000, 'mcc': '5541'},
            'GOJEK INDONESIA': {'mu': 35000, 'sigma': 15000, 'mcc': '4121'},
            'GRAB INDONESIA': {'mu': 40000, 'sigma': 15000, 'mcc': '4121'},
            'PARKIR': {'mu': 8000, 'sigma': 4000, 'mcc': '4121'},
            'CICILAN MOBIL': {'mu': 2500000, 'sigma': 500000, 'mcc': '6012'},
            'BENGKEL MOTOR': {'mu': 150000, 'sigma': 50000, 'mcc': '7538'},
            'KMT KRL COMMUTER': {'mu': 50000, 'sigma': 20000, 'mcc': '4111'},
            'E-TOLL FLAZZ': {'mu': 100000, 'sigma': 50000, 'mcc': '4784'}
        },
        'Groceries & Kebutuhan Pokok': {
            'INDOMARET': {'mu': 60000, 'sigma': 25000, 'mcc': '5411'},
            'ALFAMART': {'mu': 65000, 'sigma': 30000, 'mcc': '5411'},
            'SUPERINDO': {'mu': 300000, 'sigma': 120000, 'mcc': '5411'},
            'FOTOKOPI & PRINT': {'mu': 5000, 'sigma': 2000, 'mcc': '5111'},
            'WATSONS/GUARDIAN': {'mu': 100000, 'sigma': 50000, 'mcc': '5977'},
            'GALON & GAS': {'mu': 45000, 'sigma': 10000, 'mcc': '5499'}
        },
        'Kesehatan & Perawatan Diri': {
            'BPJS KESEHATAN': {'mu': 150000, 'sigma': 0, 'mcc': '8099'},
            'APOTEK KIMIA FARMA': {'mu': 85000, 'sigma': 30000, 'mcc': '5912'},
            'KLINIK PRATAMA': {'mu': 120000, 'sigma': 40000, 'mcc': '8099'},
            'ASURANSI JIWA': {'mu': 500000, 'sigma': 150000, 'mcc': '6300'},
            'HALODOC/ALODOKTER': {'mu': 60000, 'sigma': 20000, 'mcc': '8099'},
            'FITHUB/GYM': {'mu': 250000, 'sigma': 50000, 'mcc': '7997'}
        }
    },
    'Wants': {
        'F&B dan Nongkrong': {
            'STARBUCKS': {'mu': 75000, 'sigma': 20000, 'mcc': '5814'},
            'KOPI KENANGAN': {'mu': 35000, 'sigma': 10000, 'mcc': '5814'},
            'WARTEG': {'mu': 20000, 'sigma': 5000, 'mcc': '5812'},
            'SUSHI TEI': {'mu': 250000, 'sigma': 80000, 'mcc': '5812'},
            'ES TEH KAMPUS': {'mu': 5000, 'sigma': 1000, 'mcc': '5814'},
            'JAJANAN GEROBAK': {'mu': 10000, 'sigma': 3000, 'mcc': '5814'},
            'MIE GACOAN': {'mu': 30000, 'sigma': 15000, 'mcc': '5814'},
            'MIXUE INDONESIA': {'mu': 20000, 'sigma': 8000, 'mcc': '5814'},
            'RUMAH MAKAN PADANG': {'mu': 30000, 'sigma': 10000, 'mcc': '5812'},
            'GYU-KAKU AYCE': {'mu': 350000, 'sigma': 50000, 'mcc': '5812'},
            'CAFE AESTHETIC': {'mu': 65000, 'sigma': 20000, 'mcc': '5814'},
            'TEH TARIK PINGGIR JALAN': {'mu': 12000, 'sigma': 4000, 'mcc': '5814'},
            'NASI KUNING BU NUNUNG': {'mu': 18000, 'sigma': 5000, 'mcc': '5812'}
        },
        'Hiburan & Langganan': {
            'NETFLIX': {'mu': 186000, 'sigma': 0, 'mcc': '4899'},
            'SPOTIFY': {'mu': 54900, 'sigma': 0, 'mcc': '4899'},
            'CGV CINEMAS': {'mu': 60000, 'sigma': 15000, 'mcc': '7832'},
            'TIMEZONE': {'mu': 150000, 'sigma': 50000, 'mcc': '7999'},
            'YOUTUBE PREMIUM': {'mu': 59000, 'sigma': 0, 'mcc': '4899'},
            'TIX.ID / KONSER': {'mu': 150000, 'sigma': 50000, 'mcc': '7832'},
            'STEAM/VALORANT': {'mu': 120000, 'sigma': 50000, 'mcc': '7999'}
        },
        'Belanja Online & Fashion': {
            'SHOPEE': {'mu': 120000, 'sigma': 60000, 'mcc': '5311'},
            'TOKOPEDIA': {'mu': 150000, 'sigma': 80000, 'mcc': '5311'},
            'ZARA INDONESIA': {'mu': 500000, 'sigma': 250000, 'mcc': '5651'},
            'UNIQLO INDONESIA': {'mu': 450000, 'sigma': 150000, 'mcc': '5651'},
            'SOCIOLLA': {'mu': 250000, 'sigma': 80000, 'mcc': '5977'},
            'THRIFT SHOP IG/TIKTOK': {'mu': 100000, 'sigma': 40000, 'mcc': '5651'},
            'AKSESORIS HP ONLINE': {'mu': 30000, 'sigma': 15000, 'mcc': '5311'},
            'STICKER TELEGRAM/WA': {'mu': 10000, 'sigma': 0, 'mcc': '5734'}
        },
        'Produktivitas & Digital': {
            'CANVA PRO': {'mu': 50000, 'sigma': 0, 'mcc': '5734'},
            'NOTION': {'mu': 70000, 'sigma': 0, 'mcc': '5734'},
            'E-READER/WEBTOON': {'mu': 25000, 'sigma': 10000, 'mcc': '5734'},
            'FOTO STOCK ONLINE': {'mu': 15000, 'sigma': 5000, 'mcc': '5734'}
        }
    },
    'Transfer': {
        'Transfer P2P': {
            'REKENING TEMAN': {'mu': 50000, 'sigma': 30000, 'mcc': '4829'},
            'OVO/DANA/GOPAY': {'mu': 100000, 'sigma': 50000, 'mcc': '6540'}
        },
        'Investasi & Finansial': {
            'PT INDO PREMIER SEKURITAS': {'mu': 1000000, 'sigma': 300000, 'mcc': '6211'},
            'INDODAX': {'mu': 500000, 'sigma': 200000, 'mcc': '6211'},
            'BIBIT REKSADANA': {'mu': 300000, 'sigma': 150000, 'mcc': '6211'},
            'TOKOCRYPTO': {'mu': 250000, 'sigma': 100000, 'mcc': '6211'},
            'TABUNGAN EMAS PEGADAIAN': {'mu': 200000, 'sigma': 50000, 'mcc': '6012'},
            'MIRAE ASSET': {'mu': 1500000, 'sigma': 500000, 'mcc': '6211'},
            'STOCKBIT': {'mu': 1200000, 'sigma': 400000, 'mcc': '6211'},
            'AJAIB SEKURITAS': {'mu': 800000, 'sigma': 300000, 'mcc': '6211'},
            'REKSADANA BAREKSA': {'mu': 200000, 'sigma': 80000, 'mcc': '6211'},
            'KRIPTO PLUANG': {'mu': 300000, 'sigma': 120000, 'mcc': '6211'},
            'ORI/SUKUK RITEL': {'mu': 5000000, 'sigma': 1000000, 'mcc': '6211'},
            'EMAS DIGITAL INDOGOLD': {'mu': 150000, 'sigma': 60000, 'mcc': '6012'}
        }
    }
}

print("Master merchant siap.")

Membangun master merchant dictionary...
Master merchant siap.


# Tahap 3 — Mesin Waktu & Generator Transaksi

## Engine Transaksi

Loop 90 hari × 500 nasabah. Per hari per user:
- Gaji masuk setiap tgl 1, plus auto-debit Netflix & BPJS
- Frekuensi per kategori ditentukan `poisson_lambda_per_kategori` × persona aktif
- Dynamic throttling berlaku untuk Unconflicted & Tightwad; Spendthrift dibebaskan
- Windfall: 2% per hari mendapat pemasukan dadakan
- Cause-and-effect: persona turun jika survival streak >= 7 hari (`is_dynamic=1`)
- Output bersih: `training_data_autoencoder.csv` hanya dari `label_anomali==0` & Debit

## Tahap 3 — Running Syntax

In [15]:
print("Menjalankan engine transaksi (v2)...")

tanggal_awal   = datetime(2026, 1, 1)
total_hari     = 90
transaksi_list = []
id_trx_counter = 10001

list_windfall = ['PROFIT TRADING', 'HADIAH/GIFT', 'REFUND BELANJA', 'SIDE HUSTLE PROJECT', 'BONUS APRESIASI']

dict_p2p_ground_truth = {
    "uang kos"        : ("Needs", "Tagihan & Utilitas"),
    "patungan listrik": ("Needs", "Tagihan & Utilitas"),
    "patungan air"    : ("Needs", "Tagihan & Utilitas"),
    "beli token"      : ("Needs", "Tagihan & Utilitas"),
    "kas kelas"       : ("Needs", "Tagihan & Utilitas"),
    "fotokopi modul"  : ("Needs", "Groceries & Kebutuhan Pokok"),
    "bayar buku"      : ("Needs", "Groceries & Kebutuhan Pokok"),
    "bayar gacoan"    : ("Wants", "F&B dan Nongkrong"),
    "kopi tadi"       : ("Wants", "F&B dan Nongkrong"),
    "jajan gofood"    : ("Wants", "F&B dan Nongkrong"),
    "netflix"         : ("Wants", "Hiburan & Langganan"),
    "tiket bioskop"   : ("Wants", "Hiburan & Langganan"),
    "patungan spotify": ("Wants", "Hiburan & Langganan"),
    "utang mabar"     : ("Wants", "Hiburan & Langganan"),
    "hadiah"          : ("Wants", "Belanja Online & Fashion"),
    "beli baju"       : ("Wants", "Belanja Online & Fashion"),
    "sepatu thrifting": ("Wants", "Belanja Online & Fashion"),
    "kado ultah"      : ("Wants", "Belanja Online & Fashion"),
}

# =============================================================================
# 1. Lambda harian per kategori_detail per persona
#    Nilai mencerminkan frekuensi realistis Gen-Z Indonesia 2026
# =============================================================================
poisson_lambda_per_kategori = {
    'Spendthrift': {
        'F&B dan Nongkrong'           : 3.5,   # sarapan + makan siang + kopi + nongkrong
        'Transportasi'                : 2.0,   # ojek pergi-pulang + leisure
        'Groceries & Kebutuhan Pokok' : 1.0,   # hampir tiap hari ke minimart
        'Kesehatan & Perawatan Diri'  : 0.3,   # 2x seminggu (apotek, gym, skincare)
        'Hiburan & Langganan'         : 0.4,   # streaming + nonton + gaming
        'Belanja Online & Fashion'    : 0.5,   # impulsif 3-4x seminggu
        'Produktivitas & Digital'     : 0.15,  # ~1x seminggu
        'Tagihan & Utilitas'          : 0.10,  # 2-3x sebulan (PLN, PDAM, dll)
        'Investasi & Finansial'       : 0.15,  # nabung dikit-dikit
        'Transfer P2P'                : 1.0,   # sering kirim-terima GoPay/OVO
    },
    'Unconflicted': {
        'F&B dan Nongkrong'           : 2.0,   # makan siang + kopi
        'Transportasi'                : 1.5,   # commute harian
        'Groceries & Kebutuhan Pokok' : 0.8,   # 5-6x seminggu
        'Kesehatan & Perawatan Diri'  : 0.2,   # ~1x seminggu
        'Hiburan & Langganan'         : 0.25,  # 1-2x seminggu
        'Belanja Online & Fashion'    : 0.25,  # 1-2x seminggu
        'Produktivitas & Digital'     : 0.10,  # ~1x seminggu
        'Tagihan & Utilitas'          : 0.10,  # 2-3x sebulan
        'Investasi & Finansial'       : 0.20,  # rutin investasi
        'Transfer P2P'                : 0.6,   # beberapa kali seminggu
    },
    'Tightwad': {
        'F&B dan Nongkrong'           : 1.2,   # makan di warteg/masak sendiri
        'Transportasi'                : 1.2,   # commute tetap ada
        'Groceries & Kebutuhan Pokok' : 0.8,   # masak sendiri → belanja rutin
        'Kesehatan & Perawatan Diri'  : 0.10,  # hanya kalau perlu
        'Hiburan & Langganan'         : 0.08,  # sangat jarang
        'Belanja Online & Fashion'    : 0.07,  # sangat jarang
        'Produktivitas & Digital'     : 0.07,  # sangat jarang
        'Tagihan & Utilitas'          : 0.10,  # tetap bayar tagihan
        'Investasi & Finansial'       : 0.30,  # justru banyak investasi
        'Transfer P2P'                : 0.3,   # kadang-kadang
    },
}

# Mapping kategori_detail -> kategori_besar untuk lookup master_merchant
KATEGORI_BESAR_MAP = {
    'F&B dan Nongkrong'           : 'Wants',
    'Transportasi'                : 'Needs',
    'Groceries & Kebutuhan Pokok' : 'Needs',
    'Kesehatan & Perawatan Diri'  : 'Needs',
    'Hiburan & Langganan'         : 'Wants',
    'Belanja Online & Fashion'    : 'Wants',
    'Produktivitas & Digital'     : 'Wants',
    'Tagihan & Utilitas'          : 'Needs',
    'Investasi & Finansial'       : 'Transfer',
    'Transfer P2P'                : 'Transfer',
}

# =============================================================================
# Throttling 3-tier:
#   CORE  — kebutuhan dasar, tidak pernah 0 (makan, transport, groceries)
#   SEMI  — dikurangi di Panik, sedikit di Survival (kesehatan, hiburan, P2P, tagihan)
#   HARD  — di-block penuh saat Survival (belanja, produk digital, investasi)
#
# Multiplier per mode [Sultan, Panik, Survival]
# =============================================================================
CORE_CATS = {'F&B dan Nongkrong', 'Transportasi', 'Groceries & Kebutuhan Pokok'}
SEMI_CATS  = {'Kesehatan & Perawatan Diri', 'Hiburan & Langganan', 'Transfer P2P', 'Tagihan & Utilitas'}
HARD_CATS  = {'Belanja Online & Fashion', 'Produktivitas & Digital', 'Investasi & Finansial'}

THROTTLE = {
    'core': [1.0, 0.70, 0.40],
    'semi': [1.0, 0.50, 0.15],
    'hard': [1.0, 0.50, 0.00],
}

# Peak season hanya berlaku untuk kategori konsumtif
PEAK_CATS = {'F&B dan Nongkrong', 'Hiburan & Langganan', 'Belanja Online & Fashion', 'Transfer P2P'}

sisa_saldo_user     = {row['id_user']: 0                    for _, row in df_nasabah.iterrows()}
survival_streak     = {row['id_user']: 0                    for _, row in df_nasabah.iterrows()}
active_persona_user = {row['id_user']: row['persona_dasar'] for _, row in df_nasabah.iterrows()}


def inject_typo(text):
    if np.random.rand() < 0.15:
        noise_type = random.choice(['drop_char', 'alay_case', 'no_space'])
        if noise_type == 'drop_char' and len(text) > 4:
            idx = random.randint(0, len(text) - 1)
            return text[:idx] + text[idx + 1:]
        elif noise_type == 'alay_case':
            return "".join([c.upper() if random.random() > 0.5 else c.lower() for c in text])
        elif noise_type == 'no_space':
            return text.replace(" ", "")
    return text


def get_realistic_hour(kategori_detail, persona):
    if kategori_detail == 'F&B dan Nongkrong':
        if persona == 'Spendthrift' and np.random.rand() < 0.08:
            return random.randint(22, 23) if random.random() > 0.5 else random.randint(0, 4)
        peaks = [8, 12, 19]
        base  = random.choice(peaks)
        return min(23, max(0, base + random.randint(-1, 2)))
    elif kategori_detail == 'Hiburan & Langganan':
        return random.randint(19, 23)
    elif kategori_detail == 'Tagihan & Utilitas':
        return random.randint(8, 11)
    else:
        return random.randint(9, 21)


def process_cat(user_id, kat_detail, kat_besar, count, current_date, persona):
    global id_trx_counter
    for _ in range(count):
        mrc = random.choice(list(master_merchant[kat_besar][kat_detail].keys()))
        pm  = master_merchant[kat_besar][kat_detail][mrc]

        # Sigma cap: cegah nilai ekstrem negatif → nilai kecil tak realistis
        sigma_cap = min(pm['sigma'], pm['mu'] * 0.40) if pm['mu'] > 0 else pm['sigma']
        price     = abs(np.random.normal(pm['mu'], sigma_cap))
        price     = max(2000.0, price)  # minimum absolut Rp 2.000

        if np.random.rand() < 0.3:
            price += random.randint(1000, 6500)
        anom = 1 if np.random.rand() < 0.02 else 0
        if anom:
            price *= random.uniform(3.0, 8.0)

        if sisa_saldo_user[user_id] > price + 5000:
            sisa_saldo_user[user_id] -= price
            note      = "-"
            m_final   = mrc
            gt_besar  = kat_besar
            gt_detail = kat_detail

            if pm['mcc'] == '4829' and kat_detail != 'Investasi & Finansial':
                m_final   = f"TRF KE {fake.name().upper()}"
                kata_asli = random.choice(list(dict_p2p_ground_truth.keys()))
                gt_besar  = dict_p2p_ground_truth[kata_asli][0]
                gt_detail = dict_p2p_ground_truth[kata_asli][1]
                note      = inject_typo(kata_asli)

            hour = get_realistic_hour(kat_detail, persona)

            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=hour, minute=random.randint(0, 59)),
                'Debit', m_final, note, pm['mcc'],
                round(price, 2), round(sisa_saldo_user[user_id], 2),
                kat_besar, kat_detail, anom,
                gt_besar, gt_detail,
            ])
            id_trx_counter += 1


# =============================================================================
# LOOPING UTAMA (90 HARI)
# =============================================================================
for hari in range(total_hari):
    current_date = tanggal_awal + timedelta(days=hari)
    is_weekend   = current_date.weekday() >= 5

    peak_wants_multiplier = 1.0
    if current_date.day <= 5:
        peak_wants_multiplier = 1.5
    elif current_date.day == current_date.month:
        peak_wants_multiplier = 2.5
    if is_weekend:
        peak_wants_multiplier *= 1.3

    if current_date.day == 1:
        for index, row in df_nasabah.iterrows():
            user_id = row['id_user']
            gaji    = row['gaji_bulanan']
            sisa_saldo_user[user_id] += gaji

            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=7), 'Kredit',
                'TRF MASUK - PENDAPATAN TETAP', "-", '0000', float(gaji),
                round(sisa_saldo_user[user_id], 2),
                'Pemasukan', 'Pendapatan Bulanan', 0, None, None,
            ])
            id_trx_counter += 1

            subs = [
                ('NETFLIX',        'Wants', 'Hiburan & Langganan'),
                ('BPJS KESEHATAN', 'Needs', 'Kesehatan & Perawatan Diri'),
            ]
            for merchant_name, cat_big, cat_dtl in subs:
                p    = master_merchant[cat_big][cat_dtl][merchant_name]
                cost = p['mu']
                if sisa_saldo_user[user_id] > cost:
                    sisa_saldo_user[user_id] -= cost
                    transaksi_list.append([
                        f"TRX-{id_trx_counter}", user_id,
                        current_date.replace(hour=9), 'Debit',
                        merchant_name, "Auto-Debit Monthly", p['mcc'], float(cost),
                        round(sisa_saldo_user[user_id], 2),
                        cat_big, cat_dtl, 0, cat_big, cat_dtl,
                    ])
                    id_trx_counter += 1

            if hari > 0 and row['is_dynamic'] == 1 and survival_streak[user_id] >= 7:
                cp = active_persona_user[user_id]
                if cp == 'Spendthrift':
                    active_persona_user[user_id] = 'Unconflicted'
                elif cp == 'Unconflicted':
                    active_persona_user[user_id] = 'Tightwad'
            survival_streak[user_id] = 0
            df_nasabah.loc[
                df_nasabah['id_user'] == user_id,
                f'persona_bulan_{(hari // 30) + 1}'
            ] = active_persona_user[user_id]

    for index, row in df_nasabah.iterrows():
        user_id = row['id_user']
        gaji    = row['gaji_bulanan']

        if np.random.rand() < 0.02:
            win_val = round(abs(np.random.normal(500000, 300000)), -3)
            sisa_saldo_user[user_id] += win_val
            transaksi_list.append([
                f"TRX-{id_trx_counter}", user_id,
                current_date.replace(hour=14), 'Kredit',
                f"TRF MASUK - {random.choice(list_windfall)}", "Rejeki", '0000', float(win_val),
                round(sisa_saldo_user[user_id], 2),
                'Pemasukan', 'Pemasukan Tambahan', 0, None, None,
            ])
            id_trx_counter += 1

        curr_p = active_persona_user[user_id]
        ratio  = sisa_saldo_user[user_id] / gaji

        if ratio <= 0.15:
            survival_streak[user_id] += 1

        # Mode saldo
        if ratio > 0.4:
            mode_idx = 0   # Sultan
        elif ratio > 0.15:
            mode_idx = 1   # Panik
        else:
            mode_idx = 2   # Survival

        for kat_detail, lambda_base in poisson_lambda_per_kategori[curr_p].items():
            kat_besar = KATEGORI_BESAR_MAP[kat_detail]

            # Tier throttling
            if kat_detail in CORE_CATS:
                tier = 'core'
            elif kat_detail in SEMI_CATS:
                tier = 'semi'
            else:
                tier = 'hard'

            # Spendthrift tidak di-throttle
            throttle = 1.0 if curr_p == 'Spendthrift' else THROTTLE[tier][mode_idx]

            # Peak season hanya untuk kategori konsumtif
            peak = peak_wants_multiplier if kat_detail in PEAK_CATS else 1.0

            lam   = lambda_base * throttle * peak
            count = np.random.poisson(lam)
            if count > 0:
                process_cat(user_id, kat_detail, kat_besar, count, current_date, curr_p)


df_transaksi = pd.DataFrame(transaksi_list, columns=[
    'id_transaksi', 'id_user', 'timestamp', 'tipe_mutasi', 'deskripsi_mutasi',
    'catatan_mutasi', 'mcc', 'nominal', 'sisa_saldo', 'kategori_besar_sistem',
    'kategori_detail_sistem', 'label_anomali',
    'gt_kategori_besar', 'gt_kategori_detail',
])
df_transaksi = df_transaksi.sort_values(by=['timestamp', 'id_user']).reset_index(drop=True)
print(f"Selesai. Total transaksi: {len(df_transaksi):,}")
display(df_transaksi.head())

print("\nDistribusi kategori (Debit):")
print(df_transaksi[df_transaksi['tipe_mutasi']=='Debit']['kategori_detail_sistem'].value_counts())
print("\nMinimal nominal (Debit):", df_transaksi[df_transaksi['tipe_mutasi']=='Debit']['nominal'].min())
print("Distribusi label_anomali:", df_transaksi['label_anomali'].value_counts().to_dict())

Menjalankan engine transaksi (v2)...
Selesai. Total transaksi: 77,240


,id_transaksi,id_user,timestamp,tipe_mutasi,deskripsi_mutasi,catatan_mutasi,mcc,nominal,sisa_saldo,kategori_besar_sistem,kategori_detail_sistem,label_anomali,gt_kategori_besar,gt_kategori_detail
0,TRX-13215,USR-383,2026-01-01 00:18:00,Debit,JAJANAN GEROBAK,-,5814,10323.10,4721497.62,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
1,TRX-14783,USR-270,2026-01-01 00:20:00,Debit,MIXUE INDONESIA,-,5814,19555.28,5382262.22,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
2,TRX-15292,USR-414,2026-01-01 00:21:00,Debit,MIXUE INDONESIA,-,5814,29825.40,13033472.37,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
3,TRX-11667,USR-281,2026-01-01 00:25:00,Debit,MIXUE INDONESIA,-,5814,31952.22,4762047.78,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong
4,TRX-12935,USR-068,2026-01-01 00:39:00,Debit,WARTEG,-,5812,20160.72,1920839.28,Wants,F&B dan Nongkrong,0,Wants,F&B dan Nongkrong



Distribusi kategori (Debit):
kategori_detail_sistem
F&B dan Nongkrong              31967
Transportasi                   14880
Groceries & Kebutuhan Pokok     9274
Transfer P2P                    6552
Hiburan & Langganan             3888
Kesehatan & Perawatan Diri      2933
Belanja Online & Fashion        2479
Investasi & Finansial           1443
Produktivitas & Digital          794
Tagihan & Utilitas               670
Name: count, dtype: int64

Minimal nominal (Debit): 2000.0
Distribusi label_anomali: {0: 76101, 1: 1139}


In [16]:
display(df_nasabah[['id_user', 'persona_dasar', 'is_dynamic', 'persona_bulan_1', 'persona_bulan_2', 'persona_bulan_3']].head(20))

,id_user,persona_dasar,is_dynamic,persona_bulan_1,persona_bulan_2,persona_bulan_3
0,USR-362,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
1,USR-074,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
2,USR-375,Tightwad,0,Tightwad,Tightwad,Tightwad
3,USR-156,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
4,USR-105,Tightwad,1,Tightwad,Tightwad,Tightwad
5,USR-395,Spendthrift,0,Spendthrift,Spendthrift,Spendthrift
6,USR-378,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted
7,USR-125,Spendthrift,1,Spendthrift,Tightwad,Spendthrift
8,USR-069,Tightwad,0,Tightwad,Tightwad,Tightwad
9,USR-451,Unconflicted,0,Unconflicted,Unconflicted,Unconflicted


In [17]:
df_transaksi.to_csv(f'{OUTPUT_DIR}/df_transaksi.csv', index=False)
print(f"df_transaksi ({len(df_transaksi):,} baris) disimpan ke:")
print(f"{OUTPUT_DIR}/df_transaksi.csv")


df_training_autoencoder = df_transaksi[
    (df_transaksi['label_anomali'] == 0) &
    (df_transaksi['tipe_mutasi']   == 'Debit')
].copy() 
df_training_autoencoder.to_csv(f'{OUTPUT_DIR}/training_data_autoencoder.csv', index=False)
print(f"training_data_autoencoder ({len(df_training_autoencoder):,} baris) disimpan ke:")
print(f"{OUTPUT_DIR}/training_data_autoencoder.csv")

print(f"\nRasio anomali dalam df_transaksi:")
print(df_transaksi['label_anomali'].value_counts())
print(f"\nKategori detail dalam training set:")
print(df_training_autoencoder['kategori_detail_sistem'].value_counts())

df_transaksi (77,240 baris) disimpan ke:
/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data/new/df_transaksi.csv
training_data_autoencoder (73,741 baris) disimpan ke:
/Users/anargyaisadhimaheswara/Documents/DBS Coding Camp/CAPSTONE/AI-ENGINEER/data/new/training_data_autoencoder.csv

Rasio anomali dalam df_transaksi:
label_anomali
0    76101
1     1139
Name: count, dtype: int64

Kategori detail dalam training set:
kategori_detail_sistem
F&B dan Nongkrong              31435
Transportasi                   14657
Groceries & Kebutuhan Pokok     9149
Transfer P2P                    6437
Hiburan & Langganan             3840
Kesehatan & Perawatan Diri      2916
Belanja Online & Fashion        2444
Investasi & Finansial           1427
Produktivitas & Digital          777
Tagihan & Utilitas               659
Name: count, dtype: int64
